In [1]:
# Basic Import
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns

# Modeling
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score
from itertools import product

In [3]:
# Import CSV and convert to pandas dataframe
data_path = "/Users/toripalecek/Documents/metafer_usage/metafer_usage/cleaned_data/cleaned_metafer_usage.csv"
data = pd.read_csv(data_path)
df = pd.DataFrame(data)
df.head()

,case_number,capture_date,scans,metafer,date,time,time_bin,month,weekday,weekday_index
0,26-684_MDSDF,2026-01-02 15:17:00,6,meta 2,2026-01-02,15:17:00,15:00:00,1,Friday,4
1,26-697_AMLFA,2026-01-02 13:47:00,10,meta 2,2026-01-02,13:47:00,13:30:00,1,Friday,4
2,26-698_AMLFA,2026-01-02 13:52:00,10,meta 7,2026-01-02,13:52:00,13:30:00,1,Friday,4
3,26-1081_EOSMF,2026-01-03 13:45:00,1,meta 1,2026-01-03,13:45:00,13:30:00,1,Saturday,5
4,26-1884_AMLFA,2026-01-02 13:51:00,4,meta 3,2026-01-02,13:51:00,13:30:00,1,Friday,4


In [ ]:
# Convert time_bin to datetime and calculate operational minutes since shifts cross midnight (3:00 AM)
#time = pd.to_datetime(df['time_bin'])

#df['operational_minutes'] = (
#    time.dt.hour * 60 + time.dt.minute - 180
#) % (24*60)
#df.head()

/var/folders/rr/bmtx3y515yq966zhdj0_1rhw0000gn/T/ipykernel_26209/91763925.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  time = pd.to_datetime(df['time_bin'])


,case_number,capture_date,scans,metafer,date,time,time_bin,month,weekday,weekday_index,operational_minutes
0,26-684_MDSDF,2026-01-02 15:17:00,6,meta 2,2026-01-02,15:17:00,15:00:00,1,Friday,4,720
1,26-697_AMLFA,2026-01-02 13:47:00,10,meta 2,2026-01-02,13:47:00,13:30:00,1,Friday,4,630
2,26-698_AMLFA,2026-01-02 13:52:00,10,meta 7,2026-01-02,13:52:00,13:30:00,1,Friday,4,630
3,26-1081_EOSMF,2026-01-03 13:45:00,1,meta 1,2026-01-03,13:45:00,13:30:00,1,Saturday,5,630
4,26-1884_AMLFA,2026-01-02 13:51:00,4,meta 3,2026-01-02,13:51:00,13:30:00,1,Friday,4,630


In [4]:
# Create a custom sklearn transformer for the time column. Convert into minutes with consideration for when shift changes are.
class TimeToOperationalMinutes(BaseEstimator, TransformerMixin):
    def __init__(self, start_hour=3):
        self.start_hour = start_hour

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        time = pd.to_datetime(X.iloc[:, 0])

        operational_minutes = (
            time.dt.hour * 60 
            + time.dt.minute 
            - self.start_hour * 60
        ) % (24 * 60)

        return operational_minutes.to_numpy().reshape(-1, 1)

In [12]:
# Drop unnecessary columns for clustering
drop_cols = ['case_number','capture_date','time','month','weekday_index','date','metafer']
df_drop = df.drop(drop_cols, axis=1)
df_drop.head()

,scans,time_bin,weekday
0,6,15:00:00,Friday
1,10,13:30:00,Friday
2,10,13:30:00,Friday
3,1,13:30:00,Saturday
4,4,13:30:00,Friday


In [13]:
df_drop.shape

(6571, 3)

In [14]:
# Create a column transformer to preprocess the data for clustering
num_features = ['scans']
cat_features = ['weekday']
time_feature = ['time_bin']

numberic_transformer = StandardScaler()
oh_transformer = OneHotEncoder()

time_transformer = Pipeline(
    steps=[
        ("convert_time", TimeToOperationalMinutes()),
        ("scale_time", StandardScaler())
    ]
)
preprocessor = ColumnTransformer(
    [
        ("OneHotEncoder", oh_transformer, cat_features),
        ("Time", time_transformer, time_feature),
        ("StandardScaler", numberic_transformer, num_features)])

In [15]:
X = preprocessor.fit_transform(df_drop)

/var/folders/rr/bmtx3y515yq966zhdj0_1rhw0000gn/T/ipykernel_40481/2078213797.py:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  time = pd.to_datetime(X.iloc[:, 0])


In [27]:
# Check the shape of the transformed dataframe
X.shape

(6571, 9)

In [28]:
class DBSCANTuner:
    def __init__(self, eps=(0.2,0.4,0.6,0.8,1.0), min_samples=(4,10,20,40,80,120)):
        self.eps = eps
        self.min_samples = min_samples

    def fit(self, X):
        best_score = -1
        best_params = None
        best_model = None

        for eps, min_samples in product(self.eps, self.min_samples):
            model = DBSCAN(eps=eps, min_samples=min_samples)
            labels = model.fit_predict(X)

            # Exclude noise points for silhouette score calculation
            mask = labels != -1
            if np.sum(mask) > 1:  # Ensure there are enough points to calculate silhouette score
                score = silhouette_score(X[mask], labels[mask])
                if score > best_score:
                    best_score = score
                    best_params = {'eps': eps, 'min_samples': min_samples}
                    best_model = model

        self.best_score_ = best_score
        self.best_params_ = best_params
        self.best_model_ = best_model

In [29]:
# Train Model
dbscan_tuner = DBSCANTuner()
dbscan_tuner.fit(X)

# Access the best model and its parameters
best_model = dbscan_tuner.best_model_
best_params = dbscan_tuner.best_params_
best_score = dbscan_tuner.best_score_

print("Best Parameters:", best_params)
print("Best Silhouette Score:", best_score)

Best Parameters: {'eps': 0.2, 'min_samples': 120}
Best Silhouette Score: 0.9022601124461572


In [32]:
df["cluster"] = best_model.labels_
df.head()

,case_number,capture_date,scans,metafer,date,time,time_bin,month,weekday,weekday_index,cluster
0,26-684_MDSDF,2026-01-02 15:17:00,6,meta 2,2026-01-02,15:17:00,15:00:00,1,Friday,4,-1
1,26-697_AMLFA,2026-01-02 13:47:00,10,meta 2,2026-01-02,13:47:00,13:30:00,1,Friday,4,-1
2,26-698_AMLFA,2026-01-02 13:52:00,10,meta 7,2026-01-02,13:52:00,13:30:00,1,Friday,4,-1
3,26-1081_EOSMF,2026-01-03 13:45:00,1,meta 1,2026-01-03,13:45:00,13:30:00,1,Saturday,5,-1
4,26-1884_AMLFA,2026-01-02 13:51:00,4,meta 3,2026-01-02,13:51:00,13:30:00,1,Friday,4,-1


In [ ]:
# View the average scans per cluster, weekday, and time_bin
summary = (df.groupby(["cluster","weekday","time_bin"]).agg(avg_scans=('scans', 'mean')))
summary.head(50)

avg_scans
cluster weekday time_bin           
-1      Friday  00:00:00   6.366667
                00:30:00   6.902778
                01:00:00   6.316667
                01:30:00   6.074074
                02:00:00   5.760000
                02:30:00   3.571429
                03:00:00   5.806452
                03:30:00   4.142857
                04:00:00   4.916667
                04:30:00   5.555556
                05:00:00   9.000000
                05:30:00   4.333333
                06:00:00   3.500000
                06:30:00   6.000000
                07:00:00   7.166667
                07:30:00   9.444444
                08:00:00   7.000000
                08:30:00   7.147727
                09:00:00   6.333333
                09:30:00   5.704918
                10:00:00   6.305556
                10:30:00   4.583333
                11:00:00   4.538462
                11:30:00   5.300000
                12:00:00   4.846154
                12:30:00   5.000000
                13:00:00   5.300000
                13:30:00   5.625000
                14:00:00   5.218750
                14:30:00   5.600000
                15:00:00   6.000000
                15:30:00   5.545455
                16:00:00   5.400000
                16:30:00   5.833333
                17:00:00   5.166667
                17:30:00   4.833333
                18:00:00   6.000000
                18:30:00   6.000000
                19:00:00   6.000000
                19:30:00   6.800000
                20:00:00   5.750000
                20:30:00   6.000000
                21:00:00   6.000000
                21:30:00   6.000000
                22:00:00   6.000000
                22:30:00   6.000000
                23:00:00   6.000000
        Monday  00:00:00   5.950000
                00:30:00   7.375000
                01:00:00   6.363636

In [ ]:
# View the time range for each cluster by weekday
cluster_ranges = (
    df.groupby(["cluster", "weekday"])
      .agg(
          start_time=("time_bin", "min"),
          end_time=("time_bin", "max"),
          avg_scans=("scans", "mean"),
          observations=("scans", "count")
      )
      .reset_index()
)

print(cluster_ranges)

    cluster    weekday start_time  end_time  avg_scans  observations
0        -1     Friday   00:00:00  23:00:00   6.161324           967
1        -1     Monday   00:00:00  23:30:00   5.885149           505
2        -1   Saturday   07:00:00  22:00:00   6.045455           858
3        -1     Sunday   11:00:00  23:30:00   7.096825           630
4        -1   Thursday   00:00:00  23:30:00   6.074458          1061
5        -1    Tuesday   00:00:00  23:30:00   5.454403           636
6        -1  Wednesday   00:00:00  23:30:00   6.116854           890
7         0   Saturday   10:00:00  16:00:00   6.000000           469
8         1  Wednesday   00:00:00  02:30:00   6.000000           154
9         2   Thursday   00:00:00  23:30:00   6.000000           193
10        3     Friday   00:00:00  02:30:00   6.000000           208
